# Country / region analysis (Supplementary Appendix 6)

Added in the revision (September 2026). For each researcher in the final sample, the country of
affiliation in 2022 is looked up in OpenAlex via the crosswalk of OpenAlex author ids and Twitter
user ids by Mongeon et al. (2023; version 2 of the dataset, Mongeon et al., 2024). The affiliation
covering 2022 is used rather than the most recent one.

## Two limitations reported in the supplement

* **Retired author ids.** The crosswalk is a February 2024 snapshot and OpenAlex periodically
  merges and regenerates author ids, so a share of them no longer resolve. The unresolved ids are
  listed in `unresolved_author_ids_v2.csv`.
* **Metered API.** Since 2026 the OpenAlex API allows 1,000 free requests per day, resetting at
  midnight UTC. This job needs about 4,685 requests and therefore runs over several days on the
  free tier.

## How to run this notebook

Section 1 builds the author id list. It reloads two small files if they exist and only reads the
15 GB profile dump when they do not. The download itself is done by `fetch_openalex_v2.py`,
because it has to be resumed across days (see section 2). Sections 3 onwards run in seconds once
`openalex_results_v2.json.gz` exists.

The data files (crosswalk, author ids, OpenAlex results, per-researcher intermediates) are not
part of the repository; they live in the shared data folder given in `config.yml`
(`paths.data.processed.countries`). Only the aggregate tables and the list of unresolved ids are
written to this folder.

In [1]:
# path code directory
code_dir = "/home/lauer/projects/Mastodon/code"

# path config file
config_path = "/home/lauer/projects/Mastodon/config.yml"

import sys
import os
import json, gzip
import subprocess
import pandas as pd

sys.path.append(str(code_dir))
import utils.utils as utils

cfg = utils.load_config(config_path)
pd.set_option("display.max_rows", None)

WORK_DIR = "/home/lauer/projects/Mastodon/code/revision/r2r_check_nationalities"
# data files live in the shared data folder (config.yml: paths.data.processed.countries)
DATA_DIR = cfg["paths"]["data"]["processed"]["countries"]

# --- inputs -------------------------------------------------------------------
CROSSWALK   = os.path.join(DATA_DIR, "authors_tweeters_2024_02.csv")

# --- files written by this notebook and the fetch script -----------------------
PROFILES_SLIM    = os.path.join(DATA_DIR, "profiles_slim_v2.csv.gz")
IDS_V2           = os.path.join(DATA_DIR, "author_ids_to_fetch_v2.csv")
RESULTS_V2       = os.path.join(DATA_DIR, "openalex_results_v2.json.gz")
UNRESOLVED_V2    = os.path.join(WORK_DIR, "unresolved_author_ids_v2.csv")
PROFILES_V2      = os.path.join(DATA_DIR, "profiles_with_country_v2.csv.gz")
TABLE_COUNTRY_V2 = os.path.join(WORK_DIR, "table_s3_countries_v2.csv")
TABLE_REGION_V2  = os.path.join(WORK_DIR, "table_s3_regions_v2.csv")

FETCH_SCRIPT = os.path.join(WORK_DIR, "fetch_openalex_v2.py")

# size of the final sample of the main analysis (data/final_analyzed.csv)
N_FINAL_SAMPLE = 258409
N_MIGRATED     = 3990

## 1. Author ids to look up

Crosswalk: Mongeon, P., Bowman, T. D., Costas, R., & Arroyo-Machado, W. (2024).
*Open dataset of scholars on Twitter (X)*, version 2. https://doi.org/10.5281/zenodo.10905839

The cell reloads `profiles_slim_v2.csv.gz` and `author_ids_to_fetch_v2.csv` when they exist and
otherwise rebuilds them from the profile dump, which takes 10 to 25 minutes.

In [2]:
if os.path.exists(PROFILES_SLIM) and os.path.exists(IDS_V2):
    profiles = pd.read_csv(PROFILES_SLIM, dtype={"id": str})
    author_ids_clean = pd.read_csv(IDS_V2)["openalex_author_id"].tolist()
    print(f"loaded {len(profiles)} profiles and {len(author_ids_clean)} author ids from disk")

else:
    print("rebuilding from the profile dump - this reads a 15 GB file, 10-25 minutes")
    import utils.data_loaders as data_loaders

    profiles_researchers_df = data_loaders.load_tprofiles_prepost_si_main(cfg)
    print(f"profiles in final dataset: {len(profiles_researchers_df)} "
          f"(migrated: {int(profiles_researchers_df['is_migrated'].sum())})")

    authors_tweeters_df = pd.read_csv(CROSSWALK, dtype={"tweeter_id": str})
    matched = profiles_researchers_df.id.isin(authors_tweeters_df.tweeter_id)
    print(f"matched to an OpenAlex author id: {matched.sum()} of {len(profiles_researchers_df)} "
          f"({100 * matched.mean():.1f}%)")

    profiles_researchers_df = profiles_researchers_df[matched]
    authors_tweeters_df = authors_tweeters_df[
        authors_tweeters_df.tweeter_id.isin(profiles_researchers_df.id)
    ]

    # where a tweeter matched several author ids, prefer the primary-name match
    # (alternative == 0), then the lowest author id 
    authors_best_df = (
        authors_tweeters_df
        .sort_values(by=["tweeter_id", "alternative", "author_id"], ascending=[True, True, True])
        .drop_duplicates(subset="tweeter_id", keep="first")
    )

    profiles = (
        profiles_researchers_df
        .merge(authors_best_df[["tweeter_id", "author_id", "alternative"]],
               left_on="id", right_on="tweeter_id", how="left")
        [["id", "author_id", "alternative", "is_migrated"]]
    )

    author_ids_clean = [aid.split("/")[-1] for aid in profiles["author_id"].dropna().unique()]

    profiles.to_csv(PROFILES_SLIM, index=False, compression="gzip")
    pd.Series(author_ids_clean, name="openalex_author_id").to_csv(IDS_V2, index=False)
    print(f"saved {PROFILES_SLIM}\nsaved {IDS_V2}")

print(f"\nrows                : {len(profiles)}")
print(f"unique author ids   : {len(author_ids_clean)}")
print(f"migrants among them : {int(profiles['is_migrated'].sum())} of {N_MIGRATED} in the sample")

loaded 236322 profiles and 234244 author ids from disk

rows                : 236322
unique author ids   : 234244
migrants among them : 3701 of 3990 in the sample


## 2. Fetch author records from OpenAlex

The download is done by `fetch_openalex_v2.py` rather than in this notebook, because the free
tier allows 1,000 requests per day and the job needs about 4,685. The script can be stopped and
resumed and continues where it left off.

```
conda activate Twitter-Migration
cd /home/lauer/projects/Mastodon/code/revision/r2r_check_nationalities
python fetch_openalex_v2.py --wait         # run to completion over several days
python fetch_openalex_v2.py --status       # progress only
```

The cell below only reports progress.

In [3]:
print(subprocess.run([sys.executable, FETCH_SCRIPT, "--status"],
                     capture_output=True, text=True).stdout)

records collected : 216121
batches done      : 4685 / 4685
batches left      : 0



## 3. Check the download

Each batch should contain authors with few publications as well as prolific ones; a hard floor in
`works_count` would indicate a pagination problem in the requests.

The record count falls short of the ids requested; the difference are ids that OpenAlex has
retired since the crosswalk snapshot. This is reported as a limitation in the supplement.


In [4]:
with gzip.open(RESULTS_V2, "rt", encoding="utf-8") as f:
    results = json.load(f)

wc = sorted((r.get("works_count") or 0) for r in results)
n_low = sum(1 for w in wc if w <= 10)

print(f"ids requested    : {len(author_ids_clean)}")
print(f"records returned : {len(results)} ({100 * len(results) / len(author_ids_clean):.1f}%)")
print(f"retired ids      : {len(author_ids_clean) - len(results)} "
      f"-> {os.path.basename(UNRESOLVED_V2)}")
print(f"\nworks_count  min {wc[0]} | p10 {wc[len(wc)//10]} | median {wc[len(wc)//2]} | max {wc[-1]}")
print(f"authors with <= 10 works: {n_low} ({100 * n_low / len(wc):.1f}%)")

ids requested    : 234244
records returned : 216121 (92.3%)
retired ids      : 18123 -> unresolved_author_ids_v2.csv

works_count  min 1 | p10 10 | median 49 | max 471574
authors with <= 10 works: 24126 (11.2%)


## 4. Country of affiliation in 2022

`pick_affiliation` prefers the affiliation covering the migration year. Where an author has no
2022 affiliation on record it falls back to the closest year and flags the row, so the fallback
share can be reported. Implausible future years are ignored (OpenAlex data contain affiliations
dated 2029 and 2030).

`current_country_code` returns the most recent affiliation instead, for comparison.


In [5]:
TARGET_YEAR = 2022
MAX_PLAUSIBLE_YEAR = 2026   # ignore obvious data errors (affiliations dated 2029 / 2030 occur)


def pick_affiliation(aff_list, target=TARGET_YEAR):
    '''Country code for `target`. Returns (country_code, year_used, rule).'''
    if not isinstance(aff_list, list) or not aff_list:
        return (None, None, "no_affiliation")

    cleaned = []
    for aff in aff_list:
        years = [y for y in (aff.get("years") or []) if y <= MAX_PLAUSIBLE_YEAR]
        if years:
            cleaned.append((aff, years))
    if not cleaned:
        return (None, None, "no_usable_years")

    covering = [(a, ys) for a, ys in cleaned if target in ys]
    if covering:
        covering.sort(key=lambda t: max(t[1]), reverse=True)
        inst = covering[0][0].get("institution") or {}
        return (inst.get("country_code"), target, "covers_2022")

    aff, years = min(cleaned, key=lambda t: (min(abs(y - target) for y in t[1]), -max(t[1])))
    year_used = min(years, key=lambda y: (abs(y - target), -y))
    inst = aff.get("institution") or {}
    return (inst.get("country_code"), year_used, "nearest_year")


def current_country_code(aff_list):
    '''Most recent affiliation, for comparison with the 2022 rule.'''
    if not isinstance(aff_list, list) or not aff_list:
        return None
    best, best_year = None, -1
    for aff in aff_list:
        years = aff.get("years") or []
        if years and max(years) > best_year:
            best_year, best = max(years), aff
    if best is None:
        return None
    return (best.get("institution") or {}).get("country_code")

In [6]:
authors_meta = pd.json_normalize(results)
picked = authors_meta["affiliations"].apply(pick_affiliation)

authors_countries = pd.DataFrame({
    "author_id":        authors_meta["id"],
    "country_code":     [p[0] for p in picked],
    "affiliation_year": [p[1] for p in picked],
    "affiliation_rule": [p[2] for p in picked],
    "country_code_current": authors_meta["affiliations"].apply(current_country_code),
})

print(authors_countries["affiliation_rule"].value_counts(dropna=False), "\n")
both = (authors_countries["country_code"].notna()
        & authors_countries["country_code_current"].notna())
moved = both & (authors_countries["country_code"]
                != authors_countries["country_code_current"])
print(f"country in 2022 differs from current country: {moved.sum()} "
      f"({100 * moved.sum() / both.sum():.1f}% of authors with both)")

covers_2022       169626
nearest_year       46201
no_affiliation       294
Name: affiliation_rule, dtype: int64 

country in 2022 differs from current country: 28426 (13.2% of authors with both)


In [7]:
profiles_with_country = profiles.merge(authors_countries, on="author_id", how="left")

n_rows  = len(profiles_with_country)
n_known = profiles_with_country["country_code"].notna().sum()
n_mig_known = int(
    profiles_with_country.loc[profiles_with_country.country_code.notna(), "is_migrated"].sum()
)

print(f"rows                    : {n_rows}")
print(f"with a country          : {n_known} ({100 * n_known / n_rows:.2f}% of matched accounts)")
print(f"                          ({100 * n_known / N_FINAL_SAMPLE:.2f}% of the "
      f"{N_FINAL_SAMPLE:,} in the final sample)")
print(f"migrants with a country : {n_mig_known} of {N_MIGRATED} "
      f"({100 * n_mig_known / N_MIGRATED:.1f}%)")

profiles_with_country.to_csv(PROFILES_V2, index=False, compression="gzip")
print(f"\nsaved {PROFILES_V2}")

rows                    : 236322
with a country          : 217073 (91.85% of matched accounts)
                          (84.00% of the 258,409 in the final sample)
migrants with a country : 3402 of 3990 (85.3%)

saved /home/shared_folder/twitter_migration/data/processed/countries/profiles_with_country_v2.csv.gz


## 5. Tables for Supplementary Appendix 6

Two different percentages, reported side by side because they answer different questions:

* **`migration_rate_pct`**: of the academics affiliated in country X, what share migrated?
  These are separate populations, so this column does not sum to 100%.
* **`share_of_migrants_pct`**: of all identified migrants, what share came from country X?
  This column sums to 100%.

Read the first column against the overall migration rate in the full sample (1.5%).

In [8]:
def build_table(df, group_col, min_n=None):
    sub = df[df[group_col].notna()]
    tab = sub.groupby(group_col)["is_migrated"].agg(migrated="sum", total="size")
    tab["migrated"] = tab["migrated"].astype(int)
    tab["not_migrated"] = tab["total"] - tab["migrated"]
    tab["migration_rate_pct"]     = 100 * tab["migrated"] / tab["total"]
    tab["share_of_migrants_pct"]  = 100 * tab["migrated"] / tab["migrated"].sum()
    tab["share_of_academics_pct"] = 100 * tab["total"] / tab["total"].sum()
    tab = tab[["migrated", "not_migrated", "total", "migration_rate_pct",
               "share_of_migrants_pct", "share_of_academics_pct"]]
    if min_n is not None:
        tab = tab[tab["total"] >= min_n]
    return tab.sort_values("migration_rate_pct", ascending=False).round(2)


MIN_N = 100   # countries below this are saved but excluded from the ranking

countries_all = build_table(profiles_with_country, "country_code")
countries_all.to_csv(TABLE_COUNTRY_V2)

print(f"overall migration rate in the full sample: "
      f"{100 * N_MIGRATED / N_FINAL_SAMPLE:.2f}%\n")
print(f"countries with n >= {MIN_N}, top 15 by migration rate:")
countries_all[countries_all["total"] >= MIN_N].head(15)

overall migration rate in the full sample: 1.54%

countries with n >= 100, top 15 by migration rate:


,migrated,not_migrated,total,migration_rate_pct,share_of_migrants_pct,share_of_academics_pct
country_code,,,,,,
DE,343,9229,9572,3.58,10.08,4.41
NL,171,5153,5324,3.21,5.03,2.45
AT,44,1402,1446,3.04,1.29,0.67
CH,101,3249,3350,3.01,2.97,1.54
FR,139,5765,5904,2.35,4.09,2.72
DK,47,2163,2210,2.13,1.38,1.02
SI,4,193,197,2.03,0.12,0.09
IS,2,101,103,1.94,0.06,0.05
HU,4,217,221,1.81,0.12,0.10


In [9]:
# Region mapping.
# For the SI: Mexico is grouped with North America, Russia with Europe, Turkey and Israel
# with Asia. Countries outside the map fall into "Other".
country_to_region = {
    # Europe
    "DE": "Europe", "FR": "Europe", "IT": "Europe", "ES": "Europe",
    "NL": "Europe", "BE": "Europe", "AT": "Europe", "CH": "Europe",
    "GB": "Europe", "SE": "Europe", "NO": "Europe", "DK": "Europe",
    "FI": "Europe", "PL": "Europe", "CZ": "Europe", "PT": "Europe",
    "IE": "Europe", "GR": "Europe", "HU": "Europe", "RO": "Europe",
    "SK": "Europe", "SI": "Europe", "EE": "Europe", "LT": "Europe",
    "LV": "Europe", "BG": "Europe", "HR": "Europe", "CY": "Europe",
    "RS": "Europe", "MK": "Europe", "LU": "Europe", "IS": "Europe",
    "MT": "Europe", "UA": "Europe", "RU": "Europe",
    # North America
    "US": "North America", "CA": "North America", "MX": "North America",
    # Latin America & Caribbean
    "BR": "Latin America", "AR": "Latin America", "CL": "Latin America",
    "CO": "Latin America", "PE": "Latin America", "EC": "Latin America",
    "UY": "Latin America", "PY": "Latin America", "BO": "Latin America",
    "VE": "Latin America", "CR": "Latin America", "CU": "Latin America",
    "DO": "Latin America", "PA": "Latin America",
    # Asia
    "CN": "Asia", "JP": "Asia", "KR": "Asia", "IN": "Asia",
    "SG": "Asia", "HK": "Asia", "TW": "Asia", "TR": "Asia",
    "IR": "Asia", "IL": "Asia", "AE": "Asia", "SA": "Asia",
    "PK": "Asia", "ID": "Asia", "MY": "Asia", "BD": "Asia",
    "NP": "Asia", "TH": "Asia", "PH": "Asia", "LB": "Asia",
    "QA": "Asia", "JO": "Asia", "IQ": "Asia", "OM": "Asia",
    "VN": "Asia", "LK": "Asia", "MO": "Asia",
    # Africa
    "ZA": "Africa", "NG": "Africa", "KE": "Africa", "EG": "Africa",
    "TZ": "Africa", "GH": "Africa", "MA": "Africa", "DZ": "Africa",
    "UG": "Africa", "ET": "Africa", "TN": "Africa",
    # Oceania
    "AU": "Oceania", "NZ": "Oceania",
}


def map_region(code):
    if pd.isna(code):
        return None          # country not identified - reported separately, not as a region
    return country_to_region.get(code, "Other")


profiles_with_country["region"] = profiles_with_country["country_code"].map(map_region)

regions = build_table(profiles_with_country, "region")
regions.to_csv(TABLE_REGION_V2)

unknown = profiles_with_country["country_code"].isna()
print(f"country not identified: {unknown.sum()} people, "
      f"{int(profiles_with_country.loc[unknown, 'is_migrated'].sum())} migrants "
      f"({100 * profiles_with_country.loc[unknown, 'is_migrated'].mean():.2f}%)")
print("reported separately - it is not a region\n")
regions

country not identified: 19249 people, 299 migrants (1.55%)
reported separately - it is not a region



,migrated,not_migrated,total,migration_rate_pct,share_of_migrants_pct,share_of_academics_pct
region,,,,,,
Europe,1842,94110,95952,1.92,54.14,44.20
North America,1233,76973,78206,1.58,36.24,36.03
Oceania,181,12260,12441,1.45,5.32,5.73
Africa,17,2906,2923,0.58,0.50,1.35
Asia,100,18503,18603,0.54,2.94,8.57
Latin America,26,7775,7801,0.33,0.76,3.59
Other,3,1144,1147,0.26,0.09,0.53


---

## 6. Sensitivity checks

This section is self-contained: it defines its own helper and its own output file and nothing
above or below depends on it.

Assigning one country per person is less clear-cut than it looks. OpenAlex affiliations come
from publications, so "2022 affiliation" means the institution listed on that author's
2022-dated papers. Two consequences:

* **Dual affiliations.** About 27% of authors with a 2022 affiliation have more than one country
  that year (joint appointments, visiting positions, people mid-move). The main analysis picks
  one.
* **Which one gets picked.** The main rule takes the affiliation whose span reaches furthest
  forward. An alternative is to prefer the institution that also appears in 2021 and 2023, i.e.
  the author's stable base rather than a one-off visiting address.

The section reports three variants side by side (`table_s3_sensitivity_v2.csv`). The country
ranking is stable across them.

In [10]:
# ============================================================================
# OPTIONAL SENSITIVITY BLOCK - start   (delete to cell marked "end" to remove)
# ============================================================================
TABLE_SENSITIVITY_V2 = os.path.join(WORK_DIR, "table_s3_sensitivity_v2.csv")


def affiliation_2022_variants(aff_list, target=TARGET_YEAR, window=(2021, 2023)):
    """Both tie-breaks for the target year, plus an ambiguity flag.

    Returns (country_latest, country_stable, ambiguous):
      country_latest - affiliation whose span reaches furthest forward (the main rule)
      country_stable - affiliation that also covers `window`, i.e. the author's stable base
      ambiguous      - True if the author has more than one COUNTRY in the target year
    """
    if not isinstance(aff_list, list) or not aff_list:
        return (None, None, False)

    covering = []
    for aff in aff_list:
        years = [y for y in (aff.get("years") or []) if y <= MAX_PLAUSIBLE_YEAR]
        if target in years:
            covering.append((aff, years))
    if not covering:
        return (None, None, False)

    def cc(aff):
        return (aff.get("institution") or {}).get("country_code")

    ambiguous = len({cc(a) for a, _ in covering if cc(a)}) > 1
    latest = max(covering, key=lambda t: max(t[1]))
    stable = max(covering, key=lambda t: (sum(w in t[1] for w in window), len(t[1]), max(t[1])))
    return (cc(latest[0]), cc(stable[0]), ambiguous)


_variants = authors_meta["affiliations"].apply(affiliation_2022_variants)
variants = pd.DataFrame({
    "author_id":              authors_meta["id"],
    "country_2022_latest":    [v[0] for v in _variants],
    "country_2022_stable":    [v[1] for v in _variants],
    "country_2022_ambiguous": [v[2] for v in _variants],
})

_has = variants["country_2022_latest"].notna()
_amb = variants["country_2022_ambiguous"].sum()
_chg = (_has & (variants["country_2022_latest"] != variants["country_2022_stable"])).sum()

print(f"authors with a 2022 affiliation   : {_has.sum():,}")
print(f"  more than one country in 2022   : {_amb:,} ({100 * _amb / _has.sum():.1f}%)")
print(f"  tie-break changes the country   : {_chg:,} ({100 * _chg / _has.sum():.1f}%)")
print(f"  i.e. of the ambiguous authors   : {100 * _chg / max(_amb, 1):.1f}% are actually affected")

authors with a 2022 affiliation   : 169,298
  more than one country in 2022   : 45,113 (26.6%)
  tie-break changes the country   : 8,710 (5.1%)
  i.e. of the ambiguous authors   : 19.3% are actually affected


In [11]:
# --- the three variants, side by side ---------------------------------------
pw_sens = profiles_with_country.merge(variants, on="author_id", how="left")

tab_main   = countries_all                                    # 2022, forward-reaching tie-break
tab_stable = build_table(pw_sens, "country_2022_stable")      # 2022, stable-base tie-break
tab_unamb  = build_table(                                     # 2022, unambiguous authors only
    pw_sens[~pw_sens["country_2022_ambiguous"].fillna(False)], "country_code")

sensitivity = (
    tab_main[["total", "migration_rate_pct"]]
    .join(tab_stable[["migration_rate_pct"]].rename(
        columns={"migration_rate_pct": "rate_stable_tiebreak"}))
    .join(tab_unamb[["total", "migration_rate_pct"]].rename(
        columns={"total": "total_unambiguous",
                 "migration_rate_pct": "rate_unambiguous"}))
)
sensitivity["spread_pp"] = (
    sensitivity[["migration_rate_pct", "rate_stable_tiebreak", "rate_unambiguous"]].max(axis=1)
    - sensitivity[["migration_rate_pct", "rate_stable_tiebreak", "rate_unambiguous"]].min(axis=1)
).round(2)
sensitivity = sensitivity[sensitivity["total"] >= MIN_N].round(2)
sensitivity.to_csv(TABLE_SENSITIVITY_V2)

print("Countries ranked by the main rule; the other columns are the same figure computed")
print("differently. A small spread_pp means the choice of rule does not matter.\n")
print(f"saved {TABLE_SENSITIVITY_V2}")
sensitivity.sort_values("migration_rate_pct", ascending=False).head(15)
# ============================================================================
# OPTIONAL SENSITIVITY BLOCK - end
# ============================================================================

Countries ranked by the main rule; the other columns are the same figure computed
differently. A small spread_pp means the choice of rule does not matter.

saved /home/lauer/projects/Mastodon/code/revision/r2r_check_nationalities/table_s3_sensitivity_v2.csv


,total,migration_rate_pct,rate_stable_tiebreak,total_unambiguous,rate_unambiguous,spread_pp
country_code,,,,,,
DE,9572,3.58,3.76,7191.0,3.57,0.19
NL,5324,3.21,3.56,3868.0,3.05,0.51
AT,1446,3.04,3.57,1005.0,2.69,0.88
CH,3350,3.01,3.42,2119.0,2.60,0.82
FR,5904,2.35,2.39,4299.0,2.05,0.34
DK,2210,2.13,2.16,1395.0,2.08,0.08
SI,197,2.03,2.67,148.0,2.70,0.67
IS,103,1.94,3.61,69.0,1.45,2.16
HU,221,1.81,1.18,151.0,1.32,0.63
